# Notebook 5 - Data Validation

In this notebook I learn what data validation is and how to write validation rules in Python. I cover range, type, format, category, null, business rule, referential, date and constraint validation. At the end I put all the rules in one reusable validation function that can stop a pipeline when the data is wrong.

**Dataset:** Telco Customer Churn (7043 rows, 21 columns), after Notebooks 2, 3 and 4.

**How I test the rules:** The clean Telco data should pass all the rules, but a rule that always passes is not proven to work. So I also make a **corrupted demo copy** `df_bad`, where I inject errors of every kind and I know how many of each. A good rule must find exactly the injected number. Two topics need data that Telco does not have (dates and a second table), so I use small demo tables, and I always label them as demo.

**Rule:** `df` (raw data) is never changed.

In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")   # raw data, never modified

def fix_data_types(data):
    # Notebook 2
    out = data.copy()
    out["TotalCharges"] = pd.to_numeric(out["TotalCharges"], errors="coerce")
    out["SeniorCitizen"] = out["SeniorCitizen"].astype(bool)
    for col in ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]:
        out[col] = out[col].map({"Yes": True, "No": False})
    for col in ["gender", "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
                "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
                "Contract", "PaymentMethod", "Churn"]:
        out[col] = out[col].astype("category")
    return out

def handle_missing_values(data):
    # Notebook 3
    out = data.copy()
    new_customer = out["tenure"] == 0
    out.loc[new_customer, "TotalCharges"] = out.loc[new_customer, "TotalCharges"].fillna(0.0)
    return out

def handle_duplicates(data, key="customerID", order_col="tenure"):
    # Notebook 4
    out = data.copy()
    audit = []
    out[key] = out[key].astype(str).str.strip().str.upper()
    exact = out.duplicated(keep="first")
    audit.append(out[exact].assign(reason="exact duplicate"))
    out = out[~exact]
    older = out.sort_values(order_col, kind="stable").duplicated(subset=key, keep="last").reindex(out.index)
    audit.append(out[older].assign(reason="older version of the same " + key))
    out = out[~older]
    return out, pd.concat(audit)

df_valid, _ = handle_duplicates(handle_missing_values(fix_data_types(df)))
print("Input for validation:", df_valid.shape)

Input for validation: (7043, 21)


## 1. Problems from Sprint 4 Related to Validation

| # | Problem | Why it exists | Planned fix |
|---|---|---|---|
| 1 | In Sprint 4 I checked ranges and categories by eye, without a written rule | EDA is for exploring, not for protection | write each check as a rule in code |
| 2 | "No internet service" / "No phone service" must agree with the main service column | the same fact is stored in several columns | business rules (Section 7) |
| 3 | `TotalCharges` is zero only for customers with no tenure | structural missing, fixed in Notebook 3 | business rule (Section 7) |
| 4 | New data in the future can contain errors that are not in this file | a model in production receives new rows | one validation function in the pipeline |

## 2. What is Data Validation?

**Concept:** Data validation means checking that the data follows the rules that I defined before I use it. A rule is a statement that must be true for every row, for example "tenure is not negative". Validation does not fix anything: it **finds and reports** the rows that break a rule, and then I decide what to do with them (correct, remove, or stop the pipeline).

It is different from the earlier steps: EDA looks at the data to learn, cleaning changes the data, and validation **tests the data against written rules** and gives PASS or FAIL. The same rules can be run again on every new file, so a model never receives data that breaks them.

**Examples:**
- Simple: "age cannot be negative".
- Real-world: a web form that refuses an email without an `@`.
- Business: a bank rejects a payment for an account that does not exist.
- AI/ML: a model that receives `tenure = -5` in production gives a prediction that nobody can trust.

**Implementation:** First I build a small framework. A rule has a name, a kind, a check function that returns True for every valid row, and a severity (error or warning). A runner applies all the rules and returns a report. Then I create the corrupted demo copy `df_bad`, with known errors.

In [2]:
rules = []          # the list of all the rules for the Telco data

def add_rule(name, kind, check, severity="error"):
    rules.append({"name": name, "kind": kind, "check": check, "severity": severity})

def run_rules(data, rule_list):
    rows, failed = [], {}
    for r in rule_list:
        try:
            valid = r["check"](data).fillna(False).astype(bool)      # True = the row is valid
            bad_index = list(data.index[~valid])
            status = "PASS" if len(bad_index) == 0 else ("FAIL" if r["severity"] == "error" else "WARN")
            n_bad = len(bad_index)
        except Exception as e:
            bad_index, n_bad, status = [], np.nan, "CANNOT RUN (" + type(e).__name__ + ")"
        failed[r["name"]] = bad_index
        rows.append({"kind": r["kind"], "rule": r["name"], "severity": r["severity"],
                     "violations": n_bad, "violation_%": round(n_bad / len(data) * 100, 3) if n_bad == n_bad else np.nan,
                     "status": status})
    return pd.DataFrame(rows), failed

# ---- rule builders: each one returns a function that gives True for valid rows
def numeric_range(col, low=None, high=None):
    def check(d):
        s = d[col]
        ok = pd.Series(True, index=d.index)
        if low is not None:  ok &= (s >= low)
        if high is not None: ok &= (s <= high)
        return ok | s.isna()                      # empty values are checked by the null rules
    return check

def allowed_values(col, allowed):
    return lambda d: d[col].isin(allowed) | d[col].isna()

def matches_format(col, pattern):
    return lambda d: d[col].astype(str).str.fullmatch(pattern) | d[col].isna()

def not_null(col):
    return lambda d: d[col].notna()

def parses_as_number(col):
    return lambda d: pd.to_numeric(d[col], errors="coerce").notna() | d[col].isna()

# a first rule, to see how it works
add_rule("tenure in [0, 100]", "range", numeric_range("tenure", 0, 100))
report, failed = run_rules(df_valid, rules)
report

,kind,rule,severity,violations,violation_%,status
0,range,"tenure in [0, 100]",error,0,0.0,PASS


**What the code does:**
- `rules` is a list that stores all the rules. `add_rule` adds one rule with a name, a kind, a check function and a severity.
- Every check function receives the dataframe and returns a True/False value for **every row** (True means valid).
- `run_rules` goes through the rules and runs each check. It converts the result to a plain True/False (`fillna(False)` makes an unknown result count as a violation), finds the index of the rows that are False, and sets a status: PASS if there is no violation, FAIL for an error rule and WARN for a warning rule. If a rule cannot run (for example the column does not exist) it writes "CANNOT RUN" and does not stop the others.
- It returns a report table with the number and the percentage of violations, and a dictionary with the row index of the violations of each rule.
- The rule builders (`numeric_range`, `allowed_values`, `matches_format`, `not_null`, `parses_as_number`) are small functions that make a check function from parameters, so a rule is one line. Empty values are treated as valid by every builder except `not_null`, so one problem is reported by one rule only.

**Why this method was used:** Writing each check as a separate line in a notebook is easy, but it cannot be reused or run on a new file. A rule list can be run again on any dataframe with the same columns, and it gives the same kind of report each time. A check that returns a value per row (not a single True/False for the whole column) lets me see exactly which rows are wrong. I keep the failed row indexes, so that I can look at them and fix them. A rule that crashes should not hide the results of the other rules, which is why errors are caught and reported.

**Output meaning:** The first rule passes on the real data: no customer has a tenure outside 0 to 100.

**Effect on ML:** Every later rule is added in the same way, and at the end all of them run as one gate before the model.

In [3]:
# DEMO: a corrupted copy with known errors, to prove that the rules work
rng = np.random.default_rng(21)
df_bad = df_valid.copy()

# columns that I want to corrupt must be able to hold new text values / numbers
for col in ["gender", "Contract", "PaymentMethod", "MultipleLines", "InternetService"]:
    df_bad[col] = df_bad[col].astype(str)
df_bad["tenure"] = df_bad["tenure"].astype(float)

used = set()
injected = {}                                     # rule name -> how many errors I put in

def pick(k, mask=None):
    candidates = df_bad.index if mask is None else df_bad.index[mask]
    candidates = [i for i in candidates if i not in used]
    chosen = list(rng.choice(candidates, size=k, replace=False))
    used.update(chosen)                           # a row is never used for two kinds of error
    return chosen

# range
df_bad.loc[pick(5, df_bad["tenure"] > 0), "tenure"] = -3
df_bad.loc[pick(3, df_bad["tenure"] > 0), "tenure"] = 500
injected["tenure in [0, 100]"] = 8
df_bad.loc[pick(4), "MonthlyCharges"] = -20.0
df_bad.loc[pick(3), "MonthlyCharges"] = 999.0
injected["MonthlyCharges in [1, 200]"] = 7
df_bad.loc[pick(3, df_bad["tenure"] > 0), "TotalCharges"] = -100.0
injected["TotalCharges >= 0"] = 3

# format
bad_ids = pick(6)
df_bad.loc[bad_ids[:3], "customerID"] = "12345-ABCD"
df_bad.loc[bad_ids[3:], "customerID"] = "ABC-1234"
injected["customerID format 0000-AAAAA"] = 6

# category
df_bad.loc[pick(4), "Contract"] = "Monthly"
injected["Contract in accepted set"] = 4
df_bad.loc[pick(3), "gender"] = "M"
injected["gender in accepted set"] = 3
df_bad.loc[pick(2), "PaymentMethod"] = "Cash"
injected["PaymentMethod in accepted set"] = 2

# null
df_bad.loc[pick(1), "customerID"] = None
injected["customerID not null"] = 1
df_bad.loc[pick(3), "tenure"] = np.nan
injected["tenure not null"] = 3
df_bad.loc[pick(3), "Churn"] = np.nan
injected["Churn not null"] = 3

# business rules
df_bad.loc[pick(5, df_bad["PhoneService"] == True), "MultipleLines"] = "No phone service"
injected["PhoneService and MultipleLines agree"] = 5
df_bad.loc[pick(5, df_bad["InternetService"] != "No"), "InternetService"] = "No"
injected["InternetService and add-ons agree"] = 5
df_bad.loc[pick(3, df_bad["tenure"] == 0), "TotalCharges"] = 500.0
df_bad.loc[pick(3, df_bad["tenure"] > 0), "TotalCharges"] = 0.0
injected["TotalCharges is 0 only when tenure is 0"] = 6

# constraints
targets, sources = pick(4), pick(4)
df_bad.loc[targets, "customerID"] = df_bad.loc[sources, "customerID"].values
injected["customerID is unique"] = 4
df_bad.loc[pick(3), "tenure"] = 12.5
injected["tenure is a whole number"] = 3
df_bad.loc[pick(3), "MonthlyCharges"] = 70.123
injected["MonthlyCharges has at most 2 decimals"] = 3

print("Rows of the demo copy:", len(df_bad), "| rows changed:", len(used), "| kinds of error injected:", len(injected))

Rows of the demo copy: 7043 | rows changed: 70 | kinds of error injected: 16


**What the code does:**
- It copies the clean data into `df_bad`. Some columns are converted first, so they can accept wrong values: the category columns become plain text, and `tenure` becomes a float (to hold `NaN` and 12.5).
- The helper `pick(k, mask)` chooses `k` random rows, optionally from a group (for example only the rows with tenure above 0), and stores them in `used`, so no row gets two different errors.
- For each kind of validation it changes the chosen rows in a specific way (a negative tenure, a malformed ID, a wrong category, a missing value, a contradiction between two columns, a repeated ID, a decimal tenure), and it stores in `injected` how many errors of this kind it created, under the name of the rule that must find them.

**Why this method was used:** On real data I do not know how many errors exist, so I cannot say if a rule is correct when it finds nothing. With injected errors the answer is known, and I can test every rule: it must find exactly the number that I put in. The seed (`default_rng(21)`) makes the test repeatable. Using `used` avoids overlaps, which would make the counts different from the expected ones.

**Effect on ML:** This is the same idea as testing code with unit tests. A validation rule that was never tested may give a false feeling of safety.

## 3. Range Validation

**Concept:** Range validation checks that a numeric value is inside the minimum and maximum that are possible for it. The limits come from the business or from physics, not from the data itself.

- **When to use:** numeric columns with a natural limit (age, salary, percentage, price, tenure).
- **When not to use:** columns without a real limit; a range taken from the data (min and max of the sample) only describes the sample and will reject valid new values.
- **Advantages:** simple, fast, finds impossible values.
- **Limitations:** a value can be inside the range and still be wrong, and the limits must be chosen carefully.

**Examples:**
- Simple: age cannot be negative.
- Real-world: a percentage must be between 0 and 100.
- Business: a monthly charge must be positive and below the highest plan price.
- AI/ML: a value of 999 in a column that is normally below 120 would dominate scaling.

**Implementation:** I write the rules for the Telco data, and a small demo HR table for the examples of the sprint (age, salary, percentage). The limits are my business choices: tenure 0 to 100 months, a monthly charge from 1 to 200, and `TotalCharges` not negative.

In [4]:
add_rule("MonthlyCharges in [1, 200]", "range", numeric_range("MonthlyCharges", 1, 200))
add_rule("TotalCharges >= 0", "range", numeric_range("TotalCharges", low=0))

def compare(kind):
    selected = [r for r in rules if r["kind"] == kind]
    real, _ = run_rules(df_valid, selected)
    demo, _ = run_rules(df_bad, selected)
    out = pd.DataFrame({"rule": real["rule"], "real data": real["violations"].values, "demo copy": demo["violations"].values})
    out["injected"] = [injected.get(name, "-") for name in out["rule"]]
    out["match"] = [d == i if i != "-" else "-" for d, i in zip(out["demo copy"], out["injected"])]
    return out

print(compare("range").to_string(index=False))

# DEMO: the examples of the sprint (age, salary, percentage) on a small HR table
hr = pd.DataFrame({
    "emp_id": ["E001", "E002", "E003", "E004", "E005", "E006", "E007", "E008", "E009", "E010", "E011", "E011"],
    "age": [29, 41, -3, 35, 52, 24, 130, 38, 45, 31, 27, 27],
    "salary": [52000, 61000, 48000, -1500, 75000, 39000, 58000, 0, 83000, 67000, 45000, 45000],
    "attendance_pct": [96.5, 88.0, 101.0, 92.3, 79.5, -4.0, 85.0, 90.1, 99.9, 87.2, 94.0, 94.0],
    "gender": ["Male", "Female", "Female", "M", "Male", "Female", "Other", "Female", "male", "Female", "Male", "Male"],
    "dob": ["1995-03-14", "1983-07-22", "2001-01-01", "1989-11-05", "1972-02-18", "2000-09-30", "1900-01-01", "1986-04-12", "1979-08-25", "1993-12-02", "1997-06-09", "1997-06-09"],
    "doj": ["2019-06-01", "2010-01-15", "2020-03-01", "2008-05-20", "1999-07-01", "2024-09-15", "2020-01-01", "2012-02-30", "1975-09-01", "2013-08-12", "2021-03-01", "2021-03-01"],
})

hr_rules = []
def add_hr_rule(name, kind, check, severity="error"):
    hr_rules.append({"name": name, "kind": kind, "check": check, "severity": severity})

add_hr_rule("age in [18, 70]", "range", numeric_range("age", 18, 70))
add_hr_rule("salary >= 0", "range", numeric_range("salary", low=0))
add_hr_rule("attendance_pct in [0, 100]", "range", numeric_range("attendance_pct", 0, 100))

hr_report, hr_failed = run_rules(hr, [r for r in hr_rules if r["kind"] == "range"])
print()
print(hr_report[["rule", "violations", "status"]].to_string(index=False))
print("\nEmployees that break a range rule:", sorted(set(i for ids in hr_failed.values() for i in ids)))
print(hr.loc[sorted(set(i for ids in hr_failed.values() for i in ids)), ["emp_id", "age", "salary", "attendance_pct"]])

                      rule  real data  demo copy  injected  match
        tenure in [0, 100]          0          8         8   True
MonthlyCharges in [1, 200]          0          7         7   True
         TotalCharges >= 0          0          3         3   True

                      rule  violations status
           age in [18, 70]           2   FAIL
               salary >= 0           1   FAIL
attendance_pct in [0, 100]           2   FAIL

Employees that break a range rule: [2, 3, 5, 6]
  emp_id  age  salary  attendance_pct
2   E003   -3   48000           101.0
3   E004   35   -1500            92.3
5   E006   24   39000            -4.0
6   E007  130   58000            85.0


**What the code does:**
- It adds two more range rules for Telco with `numeric_range`, and defines `compare`, a helper that runs the rules of one kind on the real data and on the corrupted copy, and puts the number of violations next to the number that I injected, with a `match` column.
- It builds a small demo HR table with typical errors (an age of -3 and 130, a negative salary, an attendance of 101 and -4), and its own list of rules. It runs the three range rules and prints the employees that break at least one rule.

**Why this method was used:** `numeric_range` uses `>=` and `<=` so the limits are valid values. I use `low=0` only for `TotalCharges`, because there is no natural upper limit for a total. The `compare` table is the proof of the rules: if the demo count is equal to the injected count the rule works. I added the HR table because it has the exact examples of the sprint (age, salary, percentage), and Telco has no percentage column.

**Output meaning:** On the real Telco data all the range rules pass (0 violations). On the corrupted copy each rule finds exactly the injected number: 8 for tenure, 7 for monthly charges and 3 for total charges, and `match` is True. In the HR table the rules find age -3 and 130 (2), salary -1500 (1; the salary 0 is allowed because only negative is wrong), and attendance 101 and -4 (2).

**Effect on ML:** An impossible value is a very strong outlier, and it would distort the mean, the scaling and the model. Range rules catch **errors**. A value that is possible but extreme is handled with outlier methods in Notebook 6, not here.

## 4. Type Validation

**Concept:** Type validation checks that every column has the data type that I expect, and that the values can really be read as this type. There are two levels: the **schema level** (the dtype of the whole column) and the **value level** (every single value can be converted).

- **When to use:** always when a file is loaded, because file formats like CSV do not store types.
- **When not to use:** it is not needed for data that comes from a strongly typed source and was already checked.
- **Advantages:** stops type errors at the beginning, before they break a calculation.
- **Limitations:** a value with the right type can still be wrong (a number can be a valid number and a wrong amount).

**Examples:**
- Simple: a column "age" must be numeric, not text.
- Real-world: a date column must contain dates.
- Business: an amount stored as text cannot be summed.
- AI/ML: a numeric feature stored as text is treated as a category by many pipelines.

**Implementation:** I define the expected type of each Telco column (the result of Notebook 2), check the schema of the raw data and of the clean data, and add a value-level rule.

In [5]:
def kind_of(series):
    if pd.api.types.is_bool_dtype(series): return "bool"
    if pd.api.types.is_numeric_dtype(series): return "numeric"
    if isinstance(series.dtype, pd.CategoricalDtype): return "category"
    return "text"

expected_schema = {"customerID": "text", "gender": "category", "SeniorCitizen": "bool", "Partner": "bool", "Dependents": "bool",
                   "tenure": "numeric", "PhoneService": "bool", "MultipleLines": "category", "InternetService": "category",
                   "OnlineSecurity": "category", "OnlineBackup": "category", "DeviceProtection": "category", "TechSupport": "category",
                   "StreamingTV": "category", "StreamingMovies": "category", "Contract": "category", "PaperlessBilling": "bool",
                   "PaymentMethod": "category", "MonthlyCharges": "numeric", "TotalCharges": "numeric", "Churn": "category"}

def validate_schema(data, schema):
    rows = [(c, schema[c], kind_of(data[c]) if c in data.columns else "MISSING COLUMN") for c in schema]
    table = pd.DataFrame(rows, columns=["column", "expected", "found"])
    extra = [c for c in data.columns if c not in schema]
    return table[table["expected"] != table["found"]], extra

raw_wrong, raw_extra = validate_schema(df, expected_schema)
clean_wrong, clean_extra = validate_schema(df_valid, expected_schema)

print("RAW data  -> columns with a wrong type:", len(raw_wrong), "| unexpected columns:", raw_extra)
print(raw_wrong.head(6).to_string(index=False))
print("\nCLEAN data -> columns with a wrong type:", len(clean_wrong), "| unexpected columns:", clean_extra)

# value level: can every value be read as a number?
add_rule("TotalCharges can be read as a number", "type", parses_as_number("TotalCharges"))
value_rule = [r for r in rules if r["kind"] == "type"]
print("\nValue-level rule on RAW data  :", run_rules(df, value_rule)[0][["rule", "violations", "status"]].values.tolist())
print("Value-level rule on CLEAN data:", run_rules(df_valid, value_rule)[0][["rule", "violations", "status"]].values.tolist())

RAW data  -> columns with a wrong type: 18 | unexpected columns: []
       column expected   found
       gender category    text
SeniorCitizen     bool numeric
      Partner     bool    text
   Dependents     bool    text
 PhoneService     bool    text
MultipleLines category    text

CLEAN data -> columns with a wrong type: 0 | unexpected columns: []

Value-level rule on RAW data  : [['TotalCharges can be read as a number', 11, 'FAIL']]
Value-level rule on CLEAN data: [['TotalCharges can be read as a number', 0, 'PASS']]


**What the code does:**
- `kind_of` classifies a column as bool, numeric, category or text from its dtype. The bool check comes first because pandas treats bool as numeric in some functions.
- `expected_schema` is a dictionary with the expected type of each of the 21 columns (the result of the decisions in Notebook 2).
- `validate_schema` compares the expected and the found type for every column, returns only the columns that do not match, and also reports columns that are not in the schema.
- It is run on the raw data and on the clean data.
- The value-level rule `parses_as_number("TotalCharges")` converts the values with `to_numeric(errors="coerce")` and marks the values that fail. It is run on both versions.

**Why this method was used:** The schema check is very fast (21 comparisons) and tells me at once if a whole file is in the wrong format, for example when the file is loaded again without the conversion. The value-level rule is needed because the schema level only says "this column is text", and not "which values are the problem". A schema stored as a dictionary is also a document: it shows the data contract of the project. The extra-columns list protects against a new column that I did not know about.

**Output meaning:** The raw data has many columns with a wrong type (text where I expect bool, category or numeric, including `TotalCharges`). The clean data has none. The value-level rule finds 11 values in the raw `TotalCharges` that cannot be read as numbers (the blank spaces) and 0 in the clean data.

**Effect on ML:** This rule is a gate: if the pipeline receives a file with the old types, it stops before the scaler or the encoder gives a strange result.

## 5. Format Validation

**Concept:** Format validation checks that a text value follows a pattern: the right length, the right characters and the right structure. The tool for it is a **regular expression** (regex).

- **When to use:** identifiers, emails, phone numbers, postal codes, dates written as text.
- **When not to use:** free text such as comments.
- **Advantages:** catches typing errors and mixed formats.
- **Limitations:** a value with the correct format can still be wrong or not exist (a valid-looking email of nobody), and complex patterns are hard to read.

**Examples:**
- Simple: a phone number has 10 digits.
- Real-world: a postal code with 6 digits.
- Business: every customer ID has four digits, a hyphen and five capital letters.
- AI/ML: IDs with a different format can mean that data from another system was mixed in.

**Implementation:** The IDs in Telco look like `7590-VHVEG`, so the pattern is `\d{4}-[A-Z]{5}`.

In [6]:
add_rule("customerID format 0000-AAAAA", "format", matches_format("customerID", r"\d{4}-[A-Z]{5}"))

print(compare("format").to_string(index=False))

failed_ids = run_rules(df_bad, [r for r in rules if r["kind"] == "format"])[1]["customerID format 0000-AAAAA"]
print("\nIDs that break the pattern in the demo copy:")
print(df_bad.loc[failed_ids, "customerID"].tolist())

# DEMO: the same idea for a phone number and an email
contacts = pd.Series(["9876543210", "98765-43210", "987654321", "abcdefghij"], name="phone")
emails = pd.Series(["a@b.com", "ab.com", "x@y"], name="email")
print("\nPhone valid (10 digits):", contacts.str.fullmatch(r"\d{10}").tolist())
print("Email valid (text@text.text):", emails.str.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+").tolist())

                        rule  real data  demo copy  injected  match
customerID format 0000-AAAAA          0          6         6   True

IDs that break the pattern in the demo copy:
['ABC-1234', '12345-ABCD', '12345-ABCD', 'ABC-1234', '12345-ABCD', 'ABC-1234']

Phone valid (10 digits): [True, False, False, False]
Email valid (text@text.text): [True, False, False]


**What the code does:**
- `matches_format` uses `str.fullmatch`, which tests that the **whole** text follows the pattern (`match` would accept a text that only starts correctly). The pattern says: four digits (`\d{4}`), a hyphen, and five capital letters (`[A-Z]{5}`).
- The rule is added to the list, compared on the real and on the corrupted data, and the IDs that fail are printed.
- A small demo checks a phone number (exactly 10 digits) and an email (some text, an `@`, some text, a dot and some text).

**Why this method was used:** A regular expression describes the format in one line, and `fullmatch` avoids the common mistake of accepting values with extra characters at the end. I made the ID rule strict on purpose (capital letters), because in Notebook 4 I normalise the IDs to upper case, so a lower case ID means that the cleaning step was skipped. For the phone and the email I only test the structure; I cannot prove with a pattern that the address exists.

**Output meaning:** The real data passes (0 violations) and the corrupted copy finds the 6 malformed IDs (`12345-ABCD` has five digits and four letters, `ABC-1234` has the parts in the wrong order). The missing ID is not counted here, because empty values belong to the null rule. In the demo, only the first phone and the first and third emails... the printed lists show which values are valid.

**Effect on ML:** A broken format is often a sign that a part of the data comes from a different source or was typed by hand. The key for joins and duplicate checks must have one format.

## 6. Category Validation

**Concept:** Category validation checks that a categorical value belongs to the set of accepted values. The accepted set can be written in the code or, better, taken from a **reference (lookup) table**.

- **When to use:** columns with a fixed list of labels (gender, contract type, country, status).
- **When not to use:** columns where new labels are normal and welcome (product names that change every day).
- **Advantages:** finds typing errors, spelling variants and new unknown labels.
- **Limitations:** the accepted list must be maintained, and a valid label can still be the wrong one.

**Examples:**
- Simple: gender must be Male, Female or Other.
- Real-world: a country code must exist in the ISO list.
- Business: a contract type must be one of the three types that the company sells.
- AI/ML: an unknown category makes a one-hot encoder fail or create a strange column.

**Implementation:**

In [7]:
accepted = {
    "gender": ["Male", "Female"],
    "Contract": ["Month-to-month", "One year", "Two year"],
    "PaymentMethod": ["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"],
}
for col, values in accepted.items():
    add_rule(col + " in accepted set", "category", allowed_values(col, values))

# the accepted set can also be read from the data of the clean file, to see if it is complete
for col in ["Churn", "InternetService"]:
    accepted[col] = sorted(df_valid[col].dropna().astype(str).unique())
    add_rule(col + " in accepted set", "category", allowed_values(col, accepted[col]))

print(compare("category").to_string(index=False))

# DEMO: HR table - the accepted set is a lookup, and the check is case sensitive
valid_genders = ["Male", "Female", "Other"]
add_hr_rule("gender in accepted set", "category", allowed_values("gender", valid_genders))
hr_cat, hr_cat_failed = run_rules(hr, [r for r in hr_rules if r["kind"] == "category"])
print("\nHR table: values outside", valid_genders, "->", hr.loc[hr_cat_failed["gender in accepted set"], "gender"].tolist())

                           rule  real data  demo copy injected match
         gender in accepted set          0          3        3  True
       Contract in accepted set          0          4        4  True
  PaymentMethod in accepted set          0          2        2  True
          Churn in accepted set          0          0        -     -
InternetService in accepted set          0          0        -     -

HR table: values outside ['Male', 'Female', 'Other'] -> ['M', 'male']


**What the code does:**
- It defines the accepted values of three columns (from my knowledge of the business) and adds one rule per column with `allowed_values`, which uses `isin`.
- For `Churn` and `InternetService` it reads the accepted values from the clean data and stores them as a rule. This is useful as a reference for future files, but it is a weaker rule, because it only says "the same as today".
- It compares the rules on the real and on the corrupted data.
- For the HR demo it checks the gender against the list Male, Female, Other.

**Why this method was used:** `isin` compares the whole value, with the same case, so "male" and "M" are found as errors. That is what I want, because an unknown spelling would become a separate category after the encoding. For the main business columns I write the list myself, because a list taken from the data cannot find a wrong label that is already in the data. I keep the lists in one dictionary (`accepted`), so there is one place to update when the business adds a new contract type.

**Output meaning:** The real data passes all the category rules. The corrupted copy finds exactly 4 wrong contracts ("Monthly"), 3 wrong genders ("M") and 2 wrong payment methods ("Cash"). In the HR table the rule finds "M" and "male", and accepts "Other" because it is in the list.

**Effect on ML:** An unknown category in new data would make the encoder fail in production, or if it ignores it, the model would get a row with no information for that column. This rule stops it earlier and tells which value is the problem.

## 7. Null Validation

**Concept:** Null validation checks that required columns have no missing values. Which columns are required is a business decision: an ID must never be empty, but an optional comment can be.

- **When to use:** at the start (is the data complete?) and at the end of the preprocessing (is anything still missing?).
- **When not to use:** on columns where a missing value is allowed and meaningful.
- **Advantages:** simple, and it protects every later step from `NaN`.
- **Limitations:** it does not see hidden missing values like blank text, "N/A" or "?" (see Notebook 3).

**Examples:**
- Simple: the name of a customer is required.
- Real-world: a form that does not accept an empty required field.
- Business: an order without a customer ID cannot be invoiced.
- AI/ML: a scikit-learn model without an imputer stops when it receives `NaN`.

**Implementation:** After Notebook 3 no column should have a missing value, so every column is required. I add the rule for the important columns, and a rule for hidden markers.

In [8]:
for col in ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]:
    add_rule(col + " not null", "null", not_null(col))

# hidden missing values: text that is only spaces
def no_blank_text(col):
    return lambda d: ~d[col].astype(str).str.strip().isin(["", "?", "N/A", "n/a", "NA", "null", "None"])
add_rule("customerID has no blank marker", "null", no_blank_text("customerID"))

print(compare("null").to_string(index=False))

print("\nColumns with any missing value, real data:", df_valid.columns[df_valid.isna().any()].tolist())
print("Columns with any missing value, demo copy:", df_bad.columns[df_bad.isna().any()].tolist())

                          rule  real data  demo copy injected match
           customerID not null          0          1        1  True
               tenure not null          0          3        3  True
       MonthlyCharges not null          0          0        -     -
         TotalCharges not null          0          0        -     -
                Churn not null          0          3        3  True
customerID has no blank marker          0          0        -     -

Columns with any missing value, real data: []
Columns with any missing value, demo copy: ['customerID', 'tenure', 'Churn']


**What the code does:**
- It adds a `not_null` rule for five important columns using `notna()`.
- It adds a second kind of rule for hidden missing values: it converts the value to text, removes spaces and checks that it is not an empty string or a typical marker like `?` or `N/A`.
- It compares the rules, and finally lists the columns that contain any `NaN` in the real and in the corrupted data.

**Why this method was used:** The rules run on the columns that the model and the joins really need. The blank-marker rule is there because in Sprint 4 `isna()` said that `TotalCharges` was complete, while it had 11 blank strings. So a null check must look for both real and hidden empties. I keep the null rules separate from the range and format rules, as I wrote in the builders, so one error is reported once.

**Output meaning:** The real data passes (no missing value after Notebook 3). The corrupted copy finds exactly 1 empty ID, 3 empty tenure values and 3 empty Churn values. The ID that is empty does not count as a blank marker, because it is a real `NaN`, and `isna()` finds it.

**Effect on ML:** This rule is the guarantee after the imputation: the model can only train and predict if no `NaN` is left, and a new missing value in production is caught immediately.

## 8. Business Rule Validation

**Concept:** A business rule is a rule that comes from how the company works and that connects several columns. The values can be valid one by one and wrong together.

- **When to use:** columns that depend on each other.
- **When not to use:** when the relation is only a tendency and not a rule (a customer with a high tenure usually has a higher total, but not always).
- **Advantages:** finds errors that no single-column rule can find.
- **Limitations:** a rule must be agreed with the business, and a wrong rule rejects good data.

**Examples:**
- Simple: a child cannot have a driver's licence.
- Real-world: the delivery date cannot be before the order date.
- Business: a customer without internet service cannot have streaming TV.
- AI/ML: contradictions between features confuse the model and are often a sign of a bug in the data pipeline.

**Implementation:** Three rules, each with a reason, and one warning rule for a plausibility check.

| Rule | Reason |
|---|---|
| `PhoneService` is No exactly when `MultipleLines` is "No phone service" | there are no lines to count without a phone service |
| `InternetService` is No exactly when all six add-ons are "No internet service" | the add-ons need internet |
| `TotalCharges` is 0 exactly when `tenure` is 0 | a new customer has not paid yet (Notebook 3) |
| `TotalCharges` is close to `tenure x MonthlyCharges` (warning) | the total is made of the monthly bills |

In [9]:
addons = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

def phone_rule(d):
    return (d["PhoneService"] == False) == (d["MultipleLines"].astype(str) == "No phone service")

def internet_rule(d):
    no_internet = d["InternetService"].astype(str) == "No"
    return pd.concat([(d[c].astype(str) == "No internet service") == no_internet for c in addons], axis=1).all(axis=1)

def total_rule(d):
    return (d["tenure"] == 0) == (d["TotalCharges"] == 0)

def plausibility_rule(d):
    expected = d["tenure"] * d["MonthlyCharges"]
    ratio = d["TotalCharges"] / expected
    return (expected <= 0) | ratio.between(0.5, 1.5)

add_rule("PhoneService and MultipleLines agree", "business", phone_rule)
add_rule("InternetService and add-ons agree", "business", internet_rule)
add_rule("TotalCharges is 0 only when tenure is 0", "business", total_rule)
add_rule("TotalCharges close to tenure x MonthlyCharges", "business", plausibility_rule, severity="warning")

print(compare("business").to_string(index=False))

# look at the rows that break the internet rule in the demo copy
_, bad_rows = run_rules(df_bad, [r for r in rules if r["name"] == "InternetService and add-ons agree"])
print()
print(df_bad.loc[bad_rows["InternetService and add-ons agree"][:3], ["customerID", "InternetService", "OnlineSecurity", "StreamingTV"]])

                                         rule  real data  demo copy injected match
         PhoneService and MultipleLines agree          0          5        5  True
            InternetService and add-ons agree          0          5        5  True
      TotalCharges is 0 only when tenure is 0          0          6        6  True
TotalCharges close to tenure x MonthlyCharges          2         22        -     -

      customerID InternetService OnlineSecurity StreamingTV
91    2424-WVHPL              No             No          No
1176  3992-YWPKO              No             No         Yes
1941  9261-WDCAF              No             No          No


**What the code does:**
- `phone_rule` compares two True/False columns: "has no phone service" and "the label says no phone service". The rule is valid when both are the same (this is the "exactly when" logic).
- `internet_rule` does the same for the six add-on columns: it builds one True/False column for each add-on, puts them together, and uses `all(axis=1)` so the row is valid only if all six agree with `InternetService`.
- `total_rule` checks that `TotalCharges` is zero for a customer exactly when the tenure is zero.
- `plausibility_rule` divides the total by `tenure x MonthlyCharges` and accepts a ratio from 0.5 to 1.5. It is a **warning**, not an error, because prices change during the contract, and I accept a wide band.
- It runs the rules and prints a few rows that break the internet rule.

**Why this method was used:** A rule written as "A is true exactly when B is true" finds both kinds of mistake with one line (A true and B false, and A false and B true). I use the strict rules as errors, because they come from the definition of the services and from the decision of Notebook 3. The plausibility rule is a warning because I am not sure about the limits, and a warning asks a person to look and does not stop the pipeline. The `astype(str)` conversions make the rules work for both category and text columns.

**Output meaning:** On the real data the three error rules pass, which confirms the consistency that I checked in Notebook 1 and the fix of Notebook 3. The corrupted copy finds exactly 5, 5 and 6 violations. The printed rows show customers with `InternetService` No who still have an add-on "Yes" or "No" instead of "No internet service". The warning rule is not compared with an injected number: a WARN on the real data means that I should look at those customers, not that they are wrong.

**Effect on ML:** Contradictory rows teach the model wrong relations. The rule also documents the business logic of the data in code that other people can read.

## 9. Referential Validation

**Concept:** Referential validation checks that a value in one table (a **foreign key**) exists in the table that it refers to (the **parent** or the reference table). A row whose key has no parent is called an **orphan**.

- **When to use:** when data comes from several tables or files that are joined.
- **When not to use:** single-table data without any link.
- **Advantages:** finds broken links before a join loses or multiplies rows.
- **Limitations:** it needs the parent table, and it proves only that the key exists, not that it is the right one.

**Examples:**
- Simple: an order must belong to an existing customer.
- Real-world: a student's course code must be in the list of courses.
- Business: a payment of a customer that was deleted from the system.
- AI/ML: after a join, an orphan gets `NaN` for all the features of the parent table.

**Implementation:** Telco is one table, so I make a **demo `payments` table** with a foreign key `customerID`, where 7 payments belong to customers that do not exist.

In [10]:
# DEMO: a second table that refers to the customers
customers = df_valid[["customerID", "Contract"]]
ids = rng.choice(customers["customerID"].to_numpy(), size=500)
payments = pd.DataFrame({"payment_id": range(1, 501), "customerID": ids, "amount": rng.uniform(20, 120, 500).round(2)})
payments.loc[[10, 77, 150, 220, 301, 402, 480], "customerID"] = ["0000-AAAAA", "0001-BBBBB", "0002-CCCCC", "0003-DDDDD", "0004-EEEEE", "0005-FFFFF", "0006-GGGGG"]

ref_rules = [{"name": "payment customerID exists in customers", "kind": "referential", "severity": "error",
              "check": lambda d: d["customerID"].isin(customers["customerID"])}]
ref_report, ref_failed = run_rules(payments, ref_rules)
print(ref_report[["rule", "violations", "status"]].to_string(index=False))
print("Orphan payments:", payments.loc[ref_failed["payment customerID exists in customers"], ["payment_id", "customerID"]].values.tolist())

# a second way: merge with indicator, it also shows the parents that have no child
check = payments.merge(customers, on="customerID", how="outer", indicator=True)
print("\nmerge indicator:", check["_merge"].value_counts().to_dict())

# lookup (reference) table instead of a hardcoded list
contract_lookup = pd.DataFrame({"Contract": ["Month-to-month", "One year", "Two year"], "months": [1, 12, 24]})
print("\nContracts not in the lookup table:", int((~customers["Contract"].astype(str).isin(contract_lookup["Contract"])).sum()))

# the join itself can be validated: the parent key must be unique
duplicated_parent = pd.concat([customers, customers.iloc[[0]]], ignore_index=True)
try:
    payments.merge(duplicated_parent, on="customerID", validate="m:1")
except Exception as e:
    print("merge(validate='m:1') with a duplicated parent key ->", type(e).__name__)

                                  rule  violations status
payment customerID exists in customers           7   FAIL
Orphan payments: [[11, '0000-AAAAA'], [78, '0001-BBBBB'], [151, '0002-CCCCC'], [221, '0003-DDDDD'], [302, '0004-EEEEE'], [403, '0005-FFFFF'], [481, '0006-GGGGG']]

merge indicator: {'right_only': 6570, 'both': 493, 'left_only': 7}

Contracts not in the lookup table: 0
merge(validate='m:1') with a duplicated parent key -> MergeError


**What the code does:**
- It builds a demo `payments` table of 500 payments, with random existing customer IDs, and then replaces 7 of them with IDs that are not in the customer table.
- The rule uses `isin(customers["customerID"])` to check that every payment belongs to a known customer, and the runner reports the orphans.
- The second method is a `merge` with `indicator=True` and `how="outer"`: the column `_merge` says if a key is only in the left table (orphan payment), only in the right table (a customer without payment), or in both.
- The third part shows a lookup table for contract types, and checks that all contracts of the customers exist in it.
- The last part uses the `validate="m:1"` option of `merge`: it raises an error if the parent table has a repeated key, because then the join would multiply the payment rows.

**Why this method was used:** `isin` is the quickest check and gives the list of orphans. The `merge` indicator is a good second check, because it also finds the opposite problem (parents without children), which may or may not be a problem for the business. The `validate` option of `merge` is the cheapest protection of a join, because the validation is done by pandas at the moment of the join. A lookup table is better than a hardcoded list, because the business can change the table without changing the code.

**Output meaning:** The rule finds exactly the 7 orphan payments. In the merge, the 7 orphan keys appear as `left_only`, and the customers without any payment appear as `right_only` (most customers, because there are only 500 payments). The lookup finds no unknown contract. The repeated parent key raises a `MergeError`.

**Effect on ML:** After a join, an orphan row gets missing values for all the parent features, and a duplicated parent key silently adds rows. Validating the link first prevents both. The Telco churn data is one table, so this rule is a safety check for the future, when more tables (payments, support calls) are added.

## 10. Date Validation

**Concept:** Date validation checks that dates can be read, that they are possible (no 30 February), that they are in a reasonable period (not in the future, not in 1900), and that the dates of one record are in the right order.

- **When to use:** every column with dates, and especially two dates of the same record (start and end, birth and joining).
- **When not to use:** none, a date is always checked.
- **Advantages:** finds impossible dates and wrong order, which break the time based features.
- **Limitations:** a possible date can still be the wrong date, and "today" must be a fixed date for a repeatable result.

**Examples:**
- Simple: 30 February does not exist.
- Real-world: an employee cannot join before being born.
- Business: a contract cannot start in the future of the record.
- AI/ML: a wrong date gives a wrong "days since signup" feature, and a time based split with future dates causes leakage.

**Implementation:** Telco has no dates, so I use the demo HR table (`dob` = date of birth, `doj` = date of joining). The reference date is fixed to 30 June 2024.

In [11]:
REFERENCE_DATE = pd.Timestamp("2024-06-30")

hr["dob_dt"] = pd.to_datetime(hr["dob"], format="%Y-%m-%d", errors="coerce")
hr["doj_dt"] = pd.to_datetime(hr["doj"], format="%Y-%m-%d", errors="coerce")

def age_at_joining(d):
    return (d["doj_dt"] - d["dob_dt"]).dt.days / 365.25

add_hr_rule("doj is a real date", "date", lambda d: d["doj_dt"].notna())
add_hr_rule("dob is a real date", "date", lambda d: d["dob_dt"].notna())
add_hr_rule("doj is not in the future", "date", lambda d: (d["doj_dt"] <= REFERENCE_DATE) | d["doj_dt"].isna())
add_hr_rule("dob is after 1920", "date", lambda d: (d["dob_dt"] >= "1920-01-01") | d["dob_dt"].isna())
add_hr_rule("doj is after dob", "date", lambda d: (d["doj_dt"] > d["dob_dt"]) | d["doj_dt"].isna() | d["dob_dt"].isna())
add_hr_rule("age at joining >= 18", "date", lambda d: (age_at_joining(d) >= 18) | d["doj_dt"].isna() | d["dob_dt"].isna())

date_rules = [r for r in hr_rules if r["kind"] == "date"]
date_report, date_failed = run_rules(hr, date_rules)
print(date_report[["rule", "violations", "status"]].to_string(index=False))

for name, idx in date_failed.items():
    if idx:
        print("\n" + name + ":")
        print(hr.loc[idx, ["emp_id", "dob", "doj"]].to_string(index=False))

                    rule  violations status
      doj is a real date           1   FAIL
      dob is a real date           0   PASS
doj is not in the future           1   FAIL
       dob is after 1920           1   FAIL
        doj is after dob           1   FAIL
    age at joining >= 18           1   FAIL

doj is a real date:
emp_id        dob        doj
  E008 1986-04-12 2012-02-30

doj is not in the future:
emp_id        dob        doj
  E006 2000-09-30 2024-09-15

dob is after 1920:
emp_id        dob        doj
  E007 1900-01-01 2020-01-01

doj is after dob:
emp_id        dob        doj
  E009 1979-08-25 1975-09-01

age at joining >= 18:
emp_id        dob        doj
  E009 1979-08-25 1975-09-01


**What the code does:**
- It converts the two text columns to real dates with `to_datetime(format=..., errors="coerce")`. A date that does not exist (30 February 2012) becomes `NaT`.
- It adds six date rules: both dates are real dates, the joining date is not after the reference date, the birth date is not before 1920, the joining date is after the birth date, and the age at joining is at least 18 (calculated in years from the difference in days).
- Each rule that compares two dates also accepts the rows where one date is `NaT`, so a single unreadable date is reported only by the "real date" rule and not three times.
- It prints the report and the employees that break each rule.

**Why this method was used:** Reading the dates with an explicit format and `coerce` turns impossible dates into `NaT`, which I can count. A fixed `REFERENCE_DATE` makes the result repeatable, because `today()` changes every day. I check the order of the two dates and the minimum age as separate rules, because they have different reasons and different actions (a joining date before birth is a data error, while an age of 17 can also be a real exception to discuss with HR). The 1920 limit finds the default dates like 1900-01-01 that systems put when the date is unknown.

**Output meaning:** The rules find one employee for each problem: E008 has an impossible joining date (30 February), E006 joins in the future (September 2024), E007 has a birth date of 1900 (a placeholder), and E009 joins before birth (1975 against 1979), who also fails the age rule.

**Effect on ML:** Wrong dates create wrong features (negative durations, huge ages) and break time based splits. The same rules protect the future Telco data, if a signup date column is added.

## 11. Constraint Validation

**Concept:** A constraint is a structural rule of the table, in the same way as in a database: a value must be **unique**, a key must be **not null**, a value must be a **whole number**, or a value must have a maximum number of decimals. Constraints describe how the data is built, while business rules describe how the company works.

- **When to use:** keys, identifiers, counts, money.
- **When not to use:** columns where repeated values are normal (contract type).
- **Advantages:** protect the structure that joins and models depend on.
- **Limitations:** they check the shape of the data and not the meaning.

**Examples:**
- Simple: two employees cannot have the same ID.
- Real-world: the number of children is a whole number.
- Business: money has at most two decimals.
- AI/ML: a repeated key creates duplicates and train/test contamination (Notebook 4).

**Implementation:**

In [12]:
def max_decimals(col, places=2):
    factor = 10 ** places
    return lambda d: ((d[col] * factor) - (d[col] * factor).round()).abs().lt(1e-6) | d[col].isna()

add_rule("customerID is unique", "constraint", lambda d: ~d["customerID"].duplicated(keep="first") | d["customerID"].isna())
add_rule("tenure is a whole number", "constraint", lambda d: (d["tenure"] % 1 == 0) | d["tenure"].isna())
add_rule("MonthlyCharges has at most 2 decimals", "constraint", max_decimals("MonthlyCharges", 2))

print(compare("constraint").to_string(index=False))

# DEMO: HR table - a key must be unique
add_hr_rule("emp_id is unique", "constraint", lambda d: ~d["emp_id"].duplicated(keep="first"))
hr_con, hr_con_failed = run_rules(hr, [r for r in hr_rules if r["kind"] == "constraint"])
print("\nHR table:", hr_con[["rule", "violations", "status"]].values.tolist())
print("Repeated key (the later copy):", hr.loc[hr_con_failed["emp_id is unique"], "emp_id"].tolist())

                                 rule  real data  demo copy  injected  match
                 customerID is unique          0          8         4  False
             tenure is a whole number          0          3         3   True
MonthlyCharges has at most 2 decimals          0          3         3   True

HR table: [['emp_id is unique', 1, 'FAIL']]
Repeated key (the later copy): ['E011']


**What the code does:**
- The uniqueness rule uses `duplicated(keep="first")`, which marks the second and later copies of a value. It is negated, so the first appearance is valid and the repeated ones are violations. An empty ID is excluded, because it belongs to the null rule.
- The whole-number rule checks that `tenure % 1 == 0` (no decimal part).
- `max_decimals` multiplies the value by 100 and checks that it is (almost) a whole number. The tolerance `1e-6` is needed because computers store numbers like 29.85 with a tiny rounding error.
- It runs the rules on the real and on the corrupted data, and does the uniqueness test on the HR demo table.

**Why this method was used:** `keep="first"` counts the extra copies, so the number of violations is the number of repeated rows, which is what I injected (4). A direct comparison like `value * 100 == round(value * 100)` would fail on correct values because of the rounding error of floats, so I compare with a small tolerance. I wrote the constraints as rules so that they are tested with every file, and not only once when the data is loaded.

**Output meaning:** The real data passes the three constraints. The corrupted copy finds exactly 4 repeated IDs, 3 non-integer tenures (12.5) and 3 charges with three decimals (70.123). In the HR table the second `E011` is the repeated key.

**Effect on ML:** A repeated key means that one customer can be in train and in test. A decimal tenure or a charge with three decimals is a sign that the value was calculated or typed by another system.

## 12. The Complete Validation Report

**Concept:** All the rules run together in one function. This function is a **gate**: if there is a rule with status FAIL, it stops the pipeline with an error, and if there are only warnings it continues and shows them. In this way the model never trains on data that breaks a rule.

**Examples:**
- Simple: a checklist before a flight.
- Real-world: a quality check at the end of a production line.
- Business: a daily data check before the report is sent.
- AI/ML: the first step of a training or a prediction pipeline.

**Implementation:**

In [13]:
def validate_dataset(data, rule_list=None, raise_on_error=True):
    report, failed = run_rules(data, rule_list or rules)
    problems = report[report["status"].str.startswith(("FAIL", "CANNOT"))]
    if raise_on_error and len(problems) > 0:
        raise ValueError(str(len(problems)) + " validation rule(s) failed: " + ", ".join(problems["rule"]))
    return report, failed

# 1) the real data must pass
report_real, failed_real = validate_dataset(df_valid)
print("REAL DATA: rules =", len(report_real), "| PASS =", int((report_real["status"] == "PASS").sum()),
      "| WARN =", int((report_real["status"] == "WARN").sum()), "| FAIL =", int(report_real["status"].str.startswith("FAIL").sum()))

# 2) the corrupted copy must be stopped
try:
    validate_dataset(df_bad)
except ValueError as e:
    print("\nCORRUPTED COPY is stopped ->", str(e)[:200], "...")

# 3) full report of the corrupted copy
report_bad, failed_bad = validate_dataset(df_bad, raise_on_error=False)
summary = report_bad.merge(pd.Series(injected, name="injected").rename_axis("rule").reset_index(), on="rule", how="left")
summary["found = injected"] = np.where(summary["injected"].isna(), "-", summary["violations"] == summary["injected"])
print()
print(summary[["kind", "rule", "violations", "injected", "found = injected", "status"]].to_string(index=False))

print("\nRules by kind:", report_real["kind"].value_counts().to_dict())

REAL DATA: rules = 23 | PASS = 22 | WARN = 1 | FAIL = 0

CORRUPTED COPY is stopped -> 16 validation rule(s) failed: tenure in [0, 100], MonthlyCharges in [1, 200], TotalCharges >= 0, customerID format 0000-AAAAA, gender in accepted set, Contract in accepted set, PaymentMethod in accept ...

      kind                                          rule  violations  injected found = injected status
     range                            tenure in [0, 100]           8       8.0             True   FAIL
     range                    MonthlyCharges in [1, 200]           7       7.0             True   FAIL
     range                             TotalCharges >= 0           3       3.0             True   FAIL
      type          TotalCharges can be read as a number           0       NaN                -   PASS
    format                  customerID format 0000-AAAAA           6       6.0             True   FAIL
  category                        gender in accepted set           3       3.0            

**What the code does:**
- `validate_dataset` runs all the rules. It selects the rules with status FAIL or CANNOT RUN, and raises a `ValueError` with the names of the failed rules, unless `raise_on_error=False`.
- Part 1 validates the real data and prints how many rules passed, warned or failed.
- Part 2 validates the corrupted copy and catches the error, to show that the gate stops the process.
- Part 3 runs the corrupted copy without the stop, joins the report with the injected numbers, and prints one table with the number of violations, the injected number, and whether they are equal. It also counts the rules by kind.

**Why this method was used:** Raising an error is the way to make a pipeline stop automatically, because a message in a notebook can be missed, but an exception cannot. The option `raise_on_error=False` lets me look at the complete report when I want to analyse the problems. I keep warnings out of the stop condition, since they need a human decision. The joined table is the final test of the whole framework: every rule that has an injected count must find exactly that number.

**Output meaning:** The real data passes all the error rules (any WARN is for a person to look at). The corrupted copy is stopped, and the report finds exactly the injected number for every rule, so `found = injected` is True for every row that has an injected number.

**Effect on ML:** This function is the validation step of the final workflow (Notebook 16). It runs on the training data, and again on every new file before prediction.

In [14]:
report_real.to_csv("validation_report_telco.csv", index=False)
df_valid.to_csv("telco_step5_validated.csv", index=False)

if os.path.exists("data_quality_issue_log.csv"):
    log = pd.read_csv("data_quality_issue_log.csv")
    log.loc[log["id"] == 4, "status"] = "validated (Notebook 5), encoding decision in Notebook 7"
    log.to_csv("data_quality_issue_log.csv", index=False)
    print(log[["id", "column", "problem", "status"]].head(4))
else:
    print("data_quality_issue_log.csv not found (run Notebook 1 first)")

   id                           column                            problem  \
0   1                     TotalCharges                     stored as text   
1   2                     TotalCharges           11 hidden missing values   
2   3                         all rows                    duplicate check   
3   4  internet add-ons, MultipleLines  redundant 'No ... service' labels   

                                              status  
0                                  done (Notebook 2)  
1                                  done (Notebook 3)  
2                                  done (Notebook 4)  
3  validated (Notebook 5), encoding decision in N...  


**What the code does:** It saves the validation report of the real data and the validated dataset, and it updates the status of issue 4 (the repeated "No internet service" / "No phone service" labels) in the issue log.

**Why this method was used:** The saved report is the evidence for the Data Quality Summary. Issue 4 is not solved yet, because the validation only confirms that the labels are consistent. The decision to merge them or to encode them is in Notebook 7.

## 13. Documentation of the Decisions

**Decision 1 - A rule-based validation gate**

| Item | Documentation |
|---|---|
| Problem | In Sprint 4 I checked ranges, categories and relations by eye, and nothing protects the model from wrong data in the future |
| Analysis | Errors can be in one value (range, format, category, null), in a relation (business rule, date, referential) or in the structure (type, constraint), and each needs its own check |
| Technique selected | A list of rules in Python, each rule is a function that returns True for valid rows, with a runner and a gate function `validate_dataset()` |
| Reason | A rule list is repeatable, readable, and it reports the exact rows. Warnings and errors are separated, so a plausibility check does not stop the pipeline |
| Implementation | Sections 2 to 12 |
| Result | The real data passes all the error rules. The corrupted demo copy is stopped, and every rule finds exactly the number of errors that I injected |
| Impact | The model only receives data that follows the rules, and the same gate protects new data in production |

**Decision 2 - Limits and accepted values**

| Item | Documentation |
|---|---|
| Problem | A rule needs limits (range) and lists (categories) that are not in the data |
| Analysis | Limits taken from the sample only describe the sample, and would reject valid new values |
| Technique selected | Tenure 0 to 100, monthly charges 1 to 200, total charges at least 0; accepted lists written from the business definition |
| Reason | The limits are wide enough for new plans and still find impossible values (negative, 500, 999). Values that are possible but extreme are not errors and are handled in Notebook 6 |
| Implementation | Sections 3 and 6, and the dictionary `accepted` |
| Result | 0 violations on the real data |
| Impact | Impossible values are stopped, and legitimate outliers are kept for the outlier analysis |

# Summary

| Validation | What it checks | Tool in the notebook |
|---|---|---|
| Range | a number is between a minimum and a maximum | `numeric_range` |
| Type | the column dtype and each value can be read as the type | `validate_schema`, `parses_as_number` |
| Format | text follows a pattern | `str.fullmatch` and regex |
| Category | the label is in the accepted set | `isin` |
| Null | required columns have no missing value | `notna`, blank markers |
| Business rule | columns agree with each other | "A exactly when B" rules |
| Referential | a foreign key exists in the parent table | `isin`, `merge(indicator=True)`, `validate="m:1"` |
| Date | dates are real, possible, and in the right order | `to_datetime(errors="coerce")`, fixed reference date |
| Constraint | unique key, whole numbers, decimals | `duplicated`, modulo, tolerance |

**Key findings on the Telco data:**
- The clean Telco data passes all the error rules, and the rules are proven, because on the corrupted copy each one found exactly the injected number of errors.
- The raw data fails the type validation (many columns with a wrong type and 11 values of `TotalCharges` that cannot be read as numbers), which is the reason for Notebook 2.
- The business rules confirm the relation between the services, and the rule `TotalCharges` is 0 only for tenure 0 confirms the decision of Notebook 3.
- Validation reports problems, it does not fix them. The action (correct, remove, stop) is a separate decision.

**What I learned:** A check that always passes proves nothing, so a rule must be tested with data that has known errors. I also learned to separate errors (stop) from warnings (look), and to keep each rule about one thing, so that every problem is reported only once.

**Next:** Notebook 6 - Outlier Treatment.